In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
print(f"Train shape: {train.shape}")
train.head()

Train shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [3]:
freq = train['answer'].value_counts()
print(freq)
print(f"\nMost frequent: {freq.idxmax()} = {freq.max()}")
print(f"Least frequent: {freq.idxmin()} = {freq.min()}")
print(f"\nAnswer: {freq.max()} + {freq.min()} = {freq.max() + freq.min()}")

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent: B = 490
Least frequent: E = 324

Answer: 490 + 324 = 814


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

cleaned_prompts = train['prompt'].apply(clean_text)
all_words = set()
for p in cleaned_prompts:
    all_words.update(p.split())

print(f"Answer: Vocabulary size = {len(all_words)}")

Answer: Vocabulary size = 859


In [5]:
row1 = train[train['id'] == 1].iloc[0]
cleaned_row1 = clean_text(row1['prompt'])
words_row1 = cleaned_row1.split()
filtered = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]

print(f"Original words: {len(words_row1)}")
print(f"After filtering: {len(filtered)}")
print(f"Remaining words: {filtered}")
print(f"\nAnswer: {len(filtered)}")

Original words: 22
After filtering: 13
Remaining words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

Answer: 13


In [6]:
combined_texts = []
for _, row in train.iterrows():
    combined = f"{row['prompt']} {row['A']} {row['B']} {row['C']} {row['D']} {row['E']}"
    combined_texts.append(combined)

tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_texts)
vocab_size = len(tfidf.get_feature_names_out())
print(f"Answer: {vocab_size}")

Answer: 2762


In [7]:
row1_prompt_vec = tfidf.transform([row1['prompt']])
row1_a_vec = tfidf.transform([row1['A']])
sim = cosine_similarity(row1_prompt_vec, row1_a_vec)[0][0]
print(f"Answer: {sim:.4f}")

Answer: 0.2720


In [8]:
correct_count = 0
for _, row in train.iterrows():
    prompt_vec = tfidf.transform([row['prompt']])
    sims = {}
    for opt in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = tfidf.transform([row[opt]])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    if max(sims, key=sims.get) == row['answer']:
        correct_count += 1

pct = correct_count / len(train) * 100
print(f"Correct: {correct_count}/{len(train)}")
print(f"Answer: {pct:.2f}%")

Correct: 271/2000
Answer: 13.55%


In [9]:
def map_at_3(truth, preds):
    score = 0.0
    num_hits = 0
    for i, p in enumerate(preds):
        if p == truth:
            num_hits += 1
            score += num_hits / (i + 1)
    return score

q7 = map_at_3('C', ['C', 'A', 'B'])
print(f"Answer: {q7:.4f}")

Answer: 1.0000


In [10]:
q8 = map_at_3('B', ['D', 'B', 'E'])
print(f"Answer: {q8:.4f}")

Answer: 0.5000


In [11]:
freq_sorted = freq.sort_values(ascending=False)
top3 = freq_sorted.index[:3].tolist()
print(f"Top 3 most frequent: {top3}")

scores = [map_at_3(row['answer'], top3) for _, row in train.iterrows()]
print(f"\nAnswer: MAP@3 = {np.mean(scores):.4f}")

Top 3 most frequent: ['B', 'C', 'A']

Answer: MAP@3 = 0.4213


In [12]:
scores_tfidf = []
for _, row in train.iterrows():
    prompt_vec = tfidf.transform([row['prompt']])
    sims = {}
    for opt in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = tfidf.transform([row[opt]])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    scores_tfidf.append(map_at_3(row['answer'], ranked))

print(f"Answer: MAP@3 = {np.mean(scores_tfidf):.4f}")

Answer: MAP@3 = 0.2962
